<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Search a help centre or a scientific corpus

**[Search a help centre or a scientific corpus](https://learning.nextia-ai.com/courses/search/m09/search-a-help-centre-or-a-scientific-corpus/)** · Search and Ranking: Find the Best Match · Module 9, case study 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** compare BM25, dense search, two kinds of hybrid search and a cross-encoder on an open benchmark with long queries, measure them on the same queries, and decide when the cheap method is good enough.

| | |
|---|---|
| **Time** | About 60 minutes in the notebook (the whole case study: about 2.5 hours) |
| **Needs** | An internet connection for the setup cells (about 13 MB). No account and no key. The optional live cell downloads about 200 MB. |
| **You should know** | The course's measures and methods: [Ranking measures](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/), [BM25](https://learning.nextia-ai.com/courses/search/m03/bm25/), [Hybrid search](https://learning.nextia-ai.com/courses/search/m05/hybrid-search/) and [Cross-encoder re-ranking](https://learning.nextia-ai.com/courses/search/m06/cross-encoder-re-ranking/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, matplotlib 3.11.2, bm25s 0.3.16 and PyStemmer 3.1.0 (sentence-transformers 6.2.0 and PyTorch 2.15.0 for the optional live cell), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m09/search-a-help-centre-or-a-scientific-corpus/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/search-a-help-centre-or-a-scientific-corpus-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib. It takes about a minute the first time. You should see one line per package, with these versions. If you see other versions, restart the kernel, then run the cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It downloads ArguAna from the BEIR project's server (3.8 MB), and the course team's files (the split, the embeddings and the recorded model scores, about 9 MB). It checks each file's checksum, and it unzips ArguAna into the folder `arguana-search`.

You should see `Ready: 10 files in arguana-search.` If you see `Could not download`, check your internet connection (in Kaggle, turn on **Internet** in the settings). If you see `wrong checksum`, delete the folder `arguana-search` and run the cell again.

**Source and licence.** ArguAna Counterargs, by Henning Wachsmuth, Shahbaz Syed and Benno Stein (Zenodo, DOI 10.5281/zenodo.3973258), CC BY 4.0, in the BEIR version by Thakur and others. We split the queries into validation and test, and we computed the embeddings and model scores.

In [ ]:
# Setup: download the files into arguana-search/ and check each one.
import hashlib, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

FILES = {'arguana.zip': ('https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip', 'cfdf79adce27a401b3cd3ea267903134dbfab2c6afeb95d7fe5724a00bf7557b'),
    'data/split.csv': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/split.csv', 'ca74391e8197cf741c48ad0c3531afc07838e55e1c93da08c997f76e99a5cf26'),
    'data/embeddings/docs.npy': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/embeddings/docs.npy', '5902a582f1557f75ea9c4fafe919d45ffd134335bb19607e541b6fa9d92e7e58'),
    'data/embeddings/queries.npy': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/embeddings/queries.npy', '213fef379bfada8bde2b70ea818234f1c7eb538526677d476ec54c3383856273'),
    'data/embeddings/queries-128.npy': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/embeddings/queries-128.npy', '60aa25f979e7cad0a688f3f8773432683786978a0d0cafb43bb7bbbb3bc8a66d'),
    'data/embeddings/model.json': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/embeddings/model.json', '0da7dd7757837ee0f373fe316d5d94a9b49548fe06d62680f38f1502231b018d'),
    'data/recorded/rerank.csv.gz': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/recorded/rerank.csv.gz', 'd5255e12e80ef759bc9ff7530deb157b0a0e665b94c27ee5d731ea706d7a661a'),
    'data/recorded/timing.json': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/recorded/timing.json', 'a3273a0f0e135a7df8cb33adda874f4b9682f766a697029b66e6d514debdd84d'),
    'data/recorded/new-argument.npy': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L03-search-a-help-centre-or-a-scientific-corpus/data/recorded/new-argument.npy', '01a0a13770aa279e5683597aafac57b3c709ac7d668ab7ed13288a5a4b235ede'),
    'compare.py': ('https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/compare.py', 'f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2')}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():   # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("arguana-search").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:   # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name   # a copy next to the notebook, if there is one
        shutil.copy(local, path) if local.exists() else download(url, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder arguana-search and run again.")
if not (PROJECT / "arguana" / "corpus.jsonl").exists():
    zipfile.ZipFile(PROJECT / "arguana.zip").extractall(PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(f"Ready: {len(FILES)} files in arguana-search.")

## Setup: helpers

The next cell loads NumPy, pandas, matplotlib and the course's `compare.py` (as `cp`). It defines `top_ids` (the best documents, without the query's own document), `short` (the first words of a text) and the check helpers `expect` and `expect_hash`. Run it. It prints nothing.

In [ ]:
# Helpers: the data, small search helpers, and checks that do not show the answer.
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

def top_ids(scores, ids, n, drop=None):
    """The IDs of the n highest scores, best first, without the ID `drop` (the query's own document)."""
    order = np.argsort(-scores)[: n + 1]
    out = [ids[i] for i in order if ids[i] != drop]
    return out[:n]

def short(text, words=30):
    """The first words of a text, for printing."""
    w = str(text).split()
    return " ".join(w[:words]) + (" ..." if len(w) > words else "")

def fingerprint(value):
    """A short hash of a result: the checks compare it without showing the answer."""
    import hashlib
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one of your numbers with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data, briefly

Each **query** is an argument from a debate. Its one judged document is the **best counter-argument**: the answer written for the same point on the same debate page. Run the next cell.

> **Check.** You should see 8,674 documents, 1,406 queries and 1,406 judgments, one per query. Five judged documents are missing from the corpus: no method can find them.

In [ ]:
corpus = read_jsonl("arguana/corpus.jsonl")
queries = read_jsonl("arguana/queries.jsonl")
qrels = pd.read_csv("arguana/qrels/test.tsv", sep="\t")
split = pd.read_csv("data/split.csv")

doc_ids = [d["_id"] for d in corpus]
doc_text = {d["_id"]: d["text"] for d in corpus}
query_text = {q["_id"]: q["text"] for q in queries}
# One judged document per query: its best counter-argument. We call it grade 2 ("exact"), as the course does.
judgments = {q: {d: 2} for q, d in zip(qrels["query-id"], qrels["corpus-id"])}

print(f"{len(corpus):,} documents, {len(queries):,} queries, {len(qrels):,} judgments")
print("judged documents per query:", qrels.groupby("query-id").size().value_counts().to_dict())
print("judged documents missing from the corpus:", int((~qrels["corpus-id"].isin(doc_text)).sum()))
print(split["split"].value_counts().to_dict(), "| debates:", split["debate"].nunique())
print("words per query: median", int(split["words"].median()), "| words per document: median",
      int(np.median([len(t.split()) for t in doc_text.values()])))

Now read one query and its counter-argument. The counter-argument answers the query, but it uses different words.

In [ ]:
qid = "test-environment-aeghhgwpe-pro02a"
print("QUERY  ", qid, "\n", short(query_text[qid], 45), "\n")
best = next(iter(judgments[qid]))
print("ITS BEST COUNTER-ARGUMENT", best, "\n", short(doc_text[best], 45))

### First checks

> **Predict.** The documents come from the same debate pages as the queries. Will some queries also be documents?

In [ ]:
in_corpus = [q for q in query_text if q in doc_text]
same_text = [q for q in in_corpus if doc_text[q].strip() == query_text[q].strip()]
print("queries that are also documents:", len(in_corpus), "| with the same text:", len(same_text))
texts = pd.Series(list(doc_text.values()))
print("documents whose text appears twice or more:", int(texts.duplicated(keep=False).sum()))
print("empty documents:", int((texts.str.strip() == "").sum()))
by_text = {}
for d, t in doc_text.items():
    by_text.setdefault(t.strip(), []).append(d)
copies = [q for q in query_text if any(d != q for d in by_text.get(query_text[q].strip(), []))]
print("queries whose text is also a document with another ID:", len(copies))

> **Check.** 1,298 of the 1,406 queries are also in the corpus, with the same text. A search for an argument will find the argument itself first. The benchmark's rule is to leave out the query's own document. Our helper `top_ids` does that with `drop=`.

## 2. The baseline: BM25

The next cell indexes the documents with BM25 (the course's settings: k1 1.2, b 0.75, English stemming, common English words removed). It runs every query, and keeps the top 100 without the query's own document. It takes about 5 seconds.

In [ ]:
import bm25s, Stemmer
stemmer = Stemmer.Stemmer("english")
t0 = time.perf_counter()
doc_tokens = bm25s.tokenize([doc_text[d] for d in doc_ids], stopwords="en", stemmer=stemmer, show_progress=False)
bm25 = bm25s.BM25(k1=1.2, b=0.75)        # the course's settings (Lucene form)
bm25.index(doc_tokens, show_progress=False)
print(f"indexed {len(doc_ids):,} documents in {time.perf_counter() - t0:.1f} s")

def bm25_scores(text):
    """BM25 score of every document for one query (stemmed, English stopwords removed)."""
    ids = bm25.get_tokens_ids(bm25s.tokenize(text, stopwords="en", stemmer=stemmer, show_progress=False,
                                             return_ids=False)[0])
    return bm25.get_scores(ids)

def run_bm25(qids, n=100, drop_self=True):
    out, ms = {}, []
    for q in qids:
        t = time.perf_counter()
        s = bm25_scores(query_text[q])
        out[q] = top_ids(s, doc_ids, n, drop=q if drop_self else None)
        ms.append((time.perf_counter() - t) * 1000)
    return out, ms

all_q = list(split["query_id"])
val_q = list(split.loc[split["split"] == "validation", "query_id"])
test_q = list(split.loc[split["split"] == "test", "query_id"])
bm25_run, bm25_ms = run_bm25(all_q)
print(f"BM25: {np.median(bm25_ms):.1f} ms a query (p50) on this computer")

### The failure case: the query finds itself

The next cell runs BM25 on 200 test queries **without** leaving out the query's own document.

In [ ]:
self_run, _ = run_bm25(test_q[:200], n=10, drop_self=False)
with_self = cp.per_query(self_run, judgments)["ndcg@10"].mean()
without = cp.per_query({q: bm25_run[q] for q in test_q[:200]}, judgments)["ndcg@10"].mean()
first_is_self = np.mean([r[0] == q for q, r in self_run.items()])
print(f"first result is the query itself: {first_is_self:.0%}")
print(f"NDCG@10 on 200 test queries: with the query's own document {with_self:.3f}, without it {without:.3f}")

> **Look.** The query itself comes first for every query, and NDCG@10 falls a lot. The right answer can now be second at best. If a benchmark score looks too low (or too high), check what the benchmark leaves out.

Now measure BM25 on the test queries with `compare.py`:

In [ ]:
def measure(run, qids):
    """Per-query measures of a run on some queries."""
    return cp.per_query({q: run[q] for q in qids}, judgments)

pq = {"bm25": measure(bm25_run, test_q)}
print(cp.summary(pq["bm25"]))

> **Check.** NDCG@10 about 0.46, and recall@100 above 0.9: BM25 finds the counter-argument somewhere in its top 100 for most queries, but rarely first.

> **Your turn.** What share of the test queries has its best counter-argument in first place with BM25? (A query's MRR is 1 when the right document is first.)

In [ ]:
bm25_first = ...   # the share of test queries whose best counter-argument BM25 puts first (MRR is 1)
print(bm25_first)

In [ ]:
expect('bm25_first', round(bm25_first, 3) if isinstance(bm25_first, float) else bm25_first, 0.23)

## 3. Dense search

The course team encoded every document and every query with the course's bi-encoder (granite-embedding-97m-multilingual-r2), up to 512 tokens. The vectors have length 1, so a dot product is the cosine similarity.

> **Predict.** On Larkfield's product queries, dense search lost to BM25. Here the queries are long arguments. Will dense search win or lose?

In [ ]:
D = np.load("data/embeddings/docs.npy").astype(np.float32)          # one row per document, length 1
Q = np.load("data/embeddings/queries.npy").astype(np.float32)       # one row per query (first 512 tokens)
Q128 = np.load("data/embeddings/queries-128.npy").astype(np.float32)  # the same queries, cut at 128 tokens
row = {q["_id"]: i for i, q in enumerate(queries)}

def run_dense(qids, Qm=Q, n=100):
    out, ms = {}, []
    for q in qids:
        t = time.perf_counter()
        s = D @ Qm[row[q]]                 # cosine similarity with every document
        out[q] = top_ids(s, doc_ids, n, drop=q)
        ms.append((time.perf_counter() - t) * 1000)
    return out, ms

dense_run, dense_ms = run_dense(all_q)
pq["dense"] = measure(dense_run, test_q)
print(f"dense: NDCG@10 {pq['dense']['ndcg@10'].mean():.3f}; search {np.median(dense_ms):.2f} ms a query (p50), "
      "without encoding the query")

> **Check.** About 0.54: dense search wins here, by a clear margin. Long arguments say the same thing in many words, and meaning matters more than exact words.

## 4. Hybrid search

The next cell builds RRF (k 60) and a score blend, and it chooses the blend's alpha on the **validation** queries only. It takes about 20 seconds.

In [ ]:
def rrf(a, b, k=60, n=100):
    """Reciprocal rank fusion of two ranked lists."""
    s = {}
    for lst in (a, b):
        for r, d in enumerate(lst, 1):
            s[d] = s.get(d, 0) + 1 / (k + r)
    return sorted(s, key=lambda d: -s[d])[:n]

def blend(q, alpha, n=100):
    """alpha x BM25 scaled 0..1 (divided by the best) + (1 - alpha) x dense scaled 0..1 over its top 100."""
    b = bm25_scores(query_text[q]); b = b / max(b.max(), 1e-9)
    c = D @ Q[row[q]]
    top = np.sort(c)[-101]
    c = np.clip((c - top) / (c.max() - top), 0, 1)
    return top_ids(alpha * b + (1 - alpha) * c, doc_ids, n, drop=q)

rrf_run = {q: rrf(bm25_run[q], dense_run[q]) for q in all_q}
alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
alpha_val = {a: measure({q: blend(q, a) for q in val_q}, val_q)["ndcg@10"].mean() for a in alphas}
best_alpha = max(alpha_val, key=alpha_val.get)
blend_run = {q: blend(q, best_alpha) for q in all_q}
pq["hybrid-rrf"] = measure(rrf_run, test_q)
pq["hybrid-blend"] = measure(blend_run, test_q)
print("validation NDCG@10 by alpha:", {a: round(float(v), 3) for a, v in alpha_val.items()})
print(f"best alpha on validation: {best_alpha}")

> **Check.** The best alpha is 0.4: less weight on BM25 than at Larkfield (0.8), because dense search is stronger here.

> **Your turn.** RRF with k 60 is the usual setting. Try k 10 on the test queries: a smaller k gives the first ranks more weight.

In [ ]:
rrf10_run = None   # RRF of bm25_run and dense_run with k=10 instead of 60, for every test query
rrf10 = ...        # its mean NDCG@10 on the test queries, rounded to 3 decimals
print(rrf10)

In [ ]:
expect('rrf10', rrf10, 0.535, tolerance=0.002)

## 5. Re-ranking with a cross-encoder

A cross-encoder reads the query and one document together, and gives a score. The course team ran the course's cross-encoder (granite-embedding-reranker-english-r2, English) on the top 20 of the blend for every query, and recorded the scores. Running it live is slow: you see the times in part 8. The next cell chooses the depth on validation.

In [ ]:
rec = pd.read_csv("data/recorded/rerank.csv.gz")    # cross-encoder scores of the stage-one top 20 (recorded)
stage_one = rec.groupby("query_id")["doc_id"].apply(list).to_dict()   # in stage-one order
ce_score = dict(zip(zip(rec["query_id"], rec["doc_id"]), rec["score"]))

def rerank(q, depth, base):
    """Re-order the first `depth` documents of the stage-one list by the cross-encoder's score."""
    head = sorted(stage_one[q][:depth], key=lambda d: -ce_score[(q, d)])
    rest = [d for d in base[q] if d not in head]
    return head + rest

depths = [0, 5, 10, 20]
depth_val = {k: measure({q: rerank(q, k, blend_run) for q in val_q}, val_q)["ndcg@10"].mean() for k in depths}
best_depth = max(depth_val, key=depth_val.get)
print("validation NDCG@10 by depth:", {k: round(float(v), 3) for k, v in depth_val.items()})

> **Check.** The best depth on validation is 0: every depth of re-ranking makes the validation queries worse, and 20 documents are worst. The cross-encoder scores how well a document **matches** the point; here the answer is the **opposite** claim.

To be sure, measure re-ranking 10 and 20 documents on the test queries too (only to learn: the decision uses validation):

In [ ]:
for k in (10, 20):
    pq[f"rerank-{k}"] = measure({q: rerank(q, k, blend_run) for q in all_q}, test_q)

## 6. One fair comparison

Every run below uses the same 997 test queries, the same judgments and the same measure. The last three columns compare each run with BM25 on the same queries (the paired difference and its 95% interval).

In [ ]:
names = ["bm25", "dense", "hybrid-rrf", "hybrid-blend", "rerank-10", "rerank-20"]
rows = []
for n in names:
    s = cp.summary(pq[n])
    d = cp.paired(pq["bm25"], pq[n]) if n != "bm25" else None
    rows.append({"run": n, "ndcg@10": s["ndcg@10"], "low": s["ndcg@10 low"], "high": s["ndcg@10 high"],
                 "recall@100": s["recall@100"], "mrr": s["mrr"],
                 "vs bm25": None if d is None else d["difference"],
                 "vs low": None if d is None else d["low"], "vs high": None if d is None else d["high"]})
table = pd.DataFrame(rows)
print(table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

> **Look.** Read the "vs bm25" column with its interval. Then compare the blend with re-ranking: does re-ranking add anything that the interval can tell apart from zero?

## 7. Long queries change what works

The next cell cuts the queries into three equal groups by length, and measures each run on each group.

In [ ]:
cuts = split["words"].quantile([1 / 3, 2 / 3]).round().astype(int).tolist()
split["length"] = pd.cut(split["words"], [0, cuts[0], cuts[1], 10_000],
                         labels=[f"short (up to {cuts[0]} words)", f"medium ({cuts[0] + 1}-{cuts[1]})",
                                 f"long (over {cuts[1]})"])
split["topic"] = split["query_id"].str.split("-").str[1]
by_length = pd.concat({n: cp.by_slice(pq[n], split.assign(length=split["length"].astype(str)), "length")
                       .set_index("length")["ndcg@10"] for n in names}, axis=1)
print(by_length.round(3).to_string())

### Change one thing: cut the query short

Most models read only the first part of a long text. The next cell runs dense search with the queries cut at 128 tokens (the setting that Larkfield's short product queries use), and BM25 with only the first 40 words of each query.

> **Predict.** Which group loses most when the query is cut: short, medium or long queries?

In [ ]:
dense128_run, _ = run_dense(all_q, Q128)
pq["dense-128"] = measure(dense128_run, test_q)

def head_words(text, n=40):
    return " ".join(text.split()[:n])

short_bm25 = {q: top_ids(bm25_scores(head_words(query_text[q])), doc_ids, 100, drop=q) for q in test_q}
pq["bm25-first-40-words"] = measure(short_bm25, test_q)
for n in ("dense-128", "bm25-first-40-words"):
    print(n, round(pq[n]["ndcg@10"].mean(), 3))

> **Your turn.** On the long test queries, how much NDCG@10 does dense search lose when the query is cut at 128 tokens?

In [ ]:
long_q = None      # the IDs of the long test queries (column "length" starts with "long")
loss_long = ...    # dense NDCG@10 minus dense-128 NDCG@10 on those queries, rounded to 3 decimals
print(loss_long)

In [ ]:
expect('loss_long', loss_long, 0.042, tolerance=0.002)

### Worst cases by topic

The next cell measures four runs on each topic of the debates. Look at the top rows: the topics where BM25 is weakest.

In [ ]:
by_topic = pd.concat({n: cp.by_slice(pq[n], split, "topic").set_index("topic")["ndcg@10"]
                      for n in ["bm25", "dense", "hybrid-blend", "rerank-10"]}, axis=1)
by_topic["queries"] = cp.by_slice(pq["bm25"], split, "topic").set_index("topic")["queries"]
print(by_topic.sort_values("bm25").round(3).to_string())

Every method finds the right debate more often than the right answer. The next cell shows the share of the top 10 that comes from the query's own debate page.

In [ ]:
def debate(i):
    return i.rsplit("-", 1)[0]

def same_debate_share(run, qids, k=10):
    """Share of the top k that come from the query's own debate (the same topic page)."""
    return float(np.mean([np.mean([debate(d) == debate(q) for d in run[q][:k]]) for q in qids]))

def counter_first(run, qids):
    """Share of queries whose first result is a counter-point (ID ends in 'b'); the query is a point ('a')."""
    return float(np.mean([run[q][0].endswith("b") for q in qids]))

rerank10_run = {q: rerank(q, 10, blend_run) for q in test_q}
for n, run in [("bm25", bm25_run), ("dense", dense_run), ("hybrid-blend", blend_run), ("rerank-10", rerank10_run)]:
    print(f"{n:13} same debate in the top 10: {same_debate_share(run, test_q):.3f} | first result is a "
          f"counter-point: {counter_first(run, test_q):.3f} | the best counter-argument first: "
          f"{(pq[n]['mrr'] == 1).mean():.3f}")

## 8. Time and cost

The course team measured each stage live on its laptop (Apple M4, one thread, 40 test queries): BM25, encoding the query, the dense search and the cross-encoder. The next cell adds the stages for each design and computes the cost per 1,000 queries, with an assumed price of US$0.17 an hour for a 4-vCPU server.

In [ ]:
timing = json.load(open("data/recorded/timing.json"))     # measured live on the course's laptop (40 test queries)
t1 = {k: v["p50"] for k, v in timing["threads_1"].items()}
USD_PER_HOUR = 0.17     # assumption: a 4-vCPU server on demand; check your provider's price
designs = {               # milliseconds per query (p50), one thread, stages added up
    "bm25": t1["bm25"],
    "dense": t1["encode"] + t1["dense"],
    "hybrid-blend": t1["bm25"] + t1["encode"] + t1["dense"],
    "rerank-10": t1["bm25"] + t1["encode"] + t1["dense"] + t1["rerank-10"],
    "rerank-20": t1["bm25"] + t1["encode"] + t1["dense"] + t1["rerank-20"],
}
cost = pd.DataFrame({"ndcg@10": {n: pq[n]["ndcg@10"].mean() for n in designs},
                     "p50 ms": designs,
                     # 4 queries at once on the 4 vCPUs, one thread each
                     "US$ per 1,000": {n: cp.cost_per_1000(ms, USD_PER_HOUR, threads=4) for n, ms in designs.items()}})
print(cost.to_string(float_format=lambda x: f"{x:.4f}"))

> **Your turn.** What does re-ranking 10 documents change in NDCG@10, compared with the blend, per 100 ms of extra time? (A negative number means that you pay for a loss.)

In [ ]:
gain_per_ms = ...   # the change in NDCG@10 from hybrid-blend to rerank-10, per 100 ms of extra time, 4 decimals
print(gain_per_ms)

In [ ]:
expect('gain_per_ms', gain_per_ms, -0.0018, tolerance=0.01)

## 9. The chosen design on a new argument

The case study recommends the blend (alpha 0.4) without re-ranking. The next cell runs it on an argument that is not in the benchmark (written by the course team). Its embedding was recorded, so the cell needs no model.

In [ ]:
new_argument = ("Homework should be banned in primary schools. Young children learn best through play and rest, "
                "and long evenings of worksheets make them tired and unhappy. Homework also widens the gap between "
                "pupils whose parents can help them and pupils whose parents work late.")
s = bm25_scores(new_argument)
b = s / max(s.max(), 1e-9)
v = np.load("data/recorded/new-argument.npy").astype(np.float32)    # its embedding, recorded (512 tokens)
c = D @ v
top = np.sort(c)[-101]
c = np.clip((c - top) / (c.max() - top), 0, 1)
answer = top_ids(best_alpha * b + (1 - best_alpha) * c, doc_ids, 5)
for r, d in enumerate(answer, 1):
    print(r, d, "|", short(doc_text[d], 25))

> **Look.** Are the first results counter-arguments, or arguments on the same side? The benchmark's judgments cannot tell you: a person must read them.

**Optional:** the next cell can install the bi-encoder (about 400 MB with PyTorch and the model) and encode the argument live. Set `RUN_LIVE = True` first. The cosine with the recorded vector should be close to 1.

In [ ]:
# Optional: encode the new argument live. Set RUN_LIVE = True, then run the cell.
# It installs sentence-transformers and PyTorch (about 200 MB) and downloads the model (about 200 MB).
RUN_LIVE = False
if RUN_LIVE:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    "sentence-transformers==6.2.0", "torch==2.15.0"], check=True)
    import torch
    from sentence_transformers import SentenceTransformer
    meta = json.load(open("data/embeddings/model.json"))
    model = SentenceTransformer(meta["model"], revision=meta["revision"], device="cpu",
                                model_kwargs={"dtype": torch.float32})
    model.max_seq_length = 512
    live = model.encode([new_argument], normalize_embeddings=True)[0]
    print("cosine with the recorded vector:", round(float(live @ v), 4))
else:
    print("Skipped. Set RUN_LIVE = True to encode the argument live.")

## Recap

- On ArguAna, dense search beats BM25, and a blend of the two beats both. Re-ranking with the cross-encoder makes results worse and costs the most time.
- Long queries change the picture: cutting the query at 128 tokens hurts long queries most.
- Leave out the query's own document, and read the slices, not only the average.

Write your decision in the lesson's format: the design and its settings, its NDCG@10 with its interval, its time and cost per 1,000 queries, and what a person still checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/search/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m09/search-a-help-centre-or-a-scientific-corpus/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The documents, queries and judgments are ArguAna (Wachsmuth, Syed and Stein; CC BY 4.0), in the BEIR version. The split, the embeddings and the model scores were made by the course team.